# MailSleuth — train model phân loại phishing (Google Colab)

Notebook này **không chứa logic train** — nó chỉ gọi `tools/train_model.py` (nguồn sự thật duy nhất) nên chạy lại ở bất kỳ đâu cũng cho cùng kết quả.

**Quy trình:**
1. Trên máy bạn: `python tools/build_features.py --corpus datasets/raw/spamassassin --out features.jsonl`
2. Copy `train_model.py` + `features.jsonl` lên `MyDrive/mailsleuth/`
3. Ở đây: Run All → xem metrics → copy `out/*` về Drive → `tools/deploy_model.sh` trên server

> Dữ liệu đưa lên Colab **chỉ là feature vector đã khử định danh** — không có raw email.

In [ ]:
!pip -q install "scikit-learn==1.5.2" "numpy==1.26.4" "scipy==1.13.1" "joblib==1.4.2" "skl2onnx==1.16.0"
import sklearn, numpy, joblib, platform
print('python', platform.python_version(), '| sklearn', sklearn.__version__, '| numpy', numpy.__version__)

In [ ]:
import os, shutil
from google.colab import drive, files
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/mailsleuth'
os.makedirs('/content/work', exist_ok=True)
for f in ('train_model.py', 'features.jsonl'):
    src = os.path.join(SRC, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join('/content/work', f)); print('copied', f)
    else:
        print('thiếu', f, '- hãy upload thủ công bằng files.upload()')

In [ ]:
%cd /content/work
!python train_model.py --features features.jsonl --out out --model-type both --seed 42

In [ ]:
import json, pathlib
card = json.load(open('/content/work/out/model_card.json', encoding='utf-8'))
print('version   :', card['version'])
print('mẫu      :', card['nSamples'], card['labelCounts'])
print('nguồn    :', card['sources'])
print('hold-out :', card['holdout'])
print('metrics  :', json.dumps(card['metrics'], ensure_ascii=False, indent=1))
print('\n' + pathlib.Path('/content/work/out/model_card.md').read_text(encoding='utf-8')[:2000])

In [ ]:
import joblib, numpy as np
obj = joblib.load('/content/work/out/model.joblib')
m = obj['model']
print('dự đoán trên vector 0:', round(float(m.predict_proba([np.zeros(len(obj['featureNames']))])[0][1]), 4))
print('dự đoán trên vector giả (mọi feature = 1):', round(float(m.predict_proba([np.ones(len(obj['featureNames']))])[0][1]), 4))

In [ ]:
import shutil, time, os
DST = os.path.join(SRC, 'models', time.strftime('%Y%m%d-%H%M%S'))
os.makedirs(DST, exist_ok=True)
for f in os.listdir('/content/work/out'):
    shutil.copy(os.path.join('/content/work/out', f), os.path.join(DST, f))
print('→ đã lưu vào Drive:', DST)
print('Trên server: bash tools/deploy_model.sh', DST)

## Sau khi train xong

- `model.onnx` là định dạng chính cho server (chỉ cần `onnxruntime`, không phụ thuộc version scikit-learn).
- `model.joblib` là bản dự phòng (dùng khi bạn muốn train tiếp hoặc xuất ONNX từ nó).
- `model_card.json/md` là hồ sơ: nguồn dữ liệu, giấy phép, số mẫu, metrics, confusion matrix, hash — **đưa vào hồ sơ pháp y**.
- **Không** dùng notebook này lúc chạy hệ thống: Colab chỉ là nơi đào tại, server không bao giờ gọi nó.